# AF2LUMSAFE inference-strength sweep — validation only

Evaluates the completed seed-42 checkpoint at `lambda = 0, 0.25, 0.5, 0.75, 1`. No training, checkpoint mutation, test extraction, or test evaluation.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import importlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
BRANCH='codex/af2-luminance-strength-sweep'
REPO=Path('/content/coffee-bean-detection'); WORK=Path('/content')
os.chdir(WORK)
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96','gdown'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
import torch
if not torch.cuda.is_available(): raise RuntimeError('Aktifkan GPU Colab')
experiment_roots=[Path('/content/drive/MyDrive/Coffee_Bean_Detection/experiments'),*Path('/content/drive/.shortcut-targets-by-id').glob('*/Coffee_Bean_Detection/experiments')]
candidate_roots=[root/'coffee-standard-j25-af2-luminance-safe-v1' for root in experiment_roots if (root/'coffee-standard-j25-af2-luminance-safe-v1/val_reports/AF2LUMSAFE_seed42_result.json').is_file()]
if not candidate_roots: raise FileNotFoundError('AF2LUMSAFE seed-42 result tidak ditemukan di Drive')
CANDIDATE_ROOT=candidate_roots[0]; PROJECT=CANDIDATE_ROOT.parents[1]
CANDIDATE_RESULT=CANDIDATE_ROOT/'val_reports/AF2LUMSAFE_seed42_result.json'
candidate=json.loads(CANDIDATE_RESULT.read_text())
CHECKPOINT=CANDIDATE_ROOT/'AF2LUMSAFE/AF2LUMSAFE_seed42/weights/best.pt'
if not CHECKPOINT.is_file(): raise FileNotFoundError(CHECKPOINT)
print('CHECKPOINT:',CHECKPOINT,'| SHA:',candidate['checkpoint_sha256'])


In [ ]:
from coffee_detector.analysis.coffee_standard_j25_thesis_provenance import audit_j25_thesis_provenance
from coffee_detector.data.prepare_coffee_standard_j25_source_split import prepare_j25_source_split
ARCHIVE=WORK/'data_aug_11.zip'
if not ARCHIVE.is_file(): subprocess.run([sys.executable,'-m','gdown','https://drive.google.com/uc?id=1AofT7VbiNFM8ul-0vyCAKj7Rp4j5OX0f','-O',str(ARCHIVE)],check=True)
PROVENANCE=WORK/'coffee_standard_j25_thesis_provenance.json'
provenance=audit_j25_thesis_provenance(ARCHIVE,PROVENANCE)
if not provenance['decision'].startswith('PASS'): raise RuntimeError(f'Provenance gagal: {provenance["decision"]}')
DATA=WORK/'coffee-standard-j25-train-siblings-v2'
if DATA.exists(): shutil.rmtree(DATA)
contract=prepare_j25_source_split(ARCHIVE,DATA,seed=42,retain_train_siblings=True)
CONTRACT=DATA/'coffee_standard_j25_train_siblings_summary.json'
OUT=PROJECT/'experiments/coffee-standard-j25-af2-luminance-strength-sweep-v1'; OUT.mkdir(parents=True,exist_ok=True)
print('DATA:',contract['images'],'| OUT:',OUT)


In [ ]:
LOG=OUT/'strength_sweep_run.log'
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_coffee_standard_j25_af2_luminance_strength_sweep','--data-root',str(DATA),'--development-contract',str(CONTRACT),'--provenance-summary',str(PROVENANCE),'--checkpoint',str(CHECKPOINT),'--candidate-result',str(CANDIDATE_RESULT),'--output-root',str(OUT),'--device','0','--authorize-diagnostic']
print('MENJALANKAN VALIDATION-ONLY STRENGTH SWEEP | log=',LOG,flush=True)
with LOG.open('a',encoding='utf-8') as stream: process=subprocess.Popen(command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT)
last=-1
while process.poll() is None:
    completed=len(list((OUT/'strength_reports').glob('lambda_*.json'))) if (OUT/'strength_reports').is_dir() else 0
    if completed!=last: print(f'Kondisi selesai: {completed}/5',flush=True); last=completed
    time.sleep(60)
if process.returncode:
    print('\n'.join(LOG.read_text(errors='replace').splitlines()[-120:])); raise RuntimeError(f'Sweep gagal: {process.returncode}')
SUMMARY=OUT/'af2_luminance_strength_sweep.json'
result=json.loads(SUMMARY.read_text())
print('INTERPRETATION:',result['interpretation'])
print('PARETO:',result['pareto_frontier'])
print('TARGET RESCUED:',result['target_class_rescued_strengths'])
print('TRAINING:',result['training_executed'],'| TEST:',result['test_opened'])


In [ ]:
import pandas as pd
table=pd.DataFrame(result['values']).T
table.index.name='lambda'
display(table.style.format('{:.2%}'))
print('METRIC WINNERS:',result['metric_winners'])
print('TARGET WINNER:',result['target_class_winner'])
print('SUMMARY:',SUMMARY)
